In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import warnings

sns.set()
warnings.filterwarnings('ignore')

In [2]:
data = pd.read_csv('../data/gym_churn_us_dummy.csv')

In [3]:
data.head()

,near_location,partner,promo_friends,group_visits,avg_additional_charges_total,lifetime,churn,frequency_drop,contract_period_6,contract_period_12,age_group_26-35,age_group_36+
0,1,1,1,1,14.227470,3,0,0.020398,1,0,1,0
1,1,0,0,1,113.202938,7,0,0.012693,0,1,1,0
2,1,1,0,0,129.448479,2,0,0.122596,0,0,1,0
3,1,1,1,1,62.669863,2,0,-0.151582,0,1,1,0
4,1,1,1,0,198.362265,3,0,-0.006194,0,0,1,0


In [8]:
X = data.drop(columns=['churn'])
y = data['churn']

Train test split

In [9]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [10]:
x_train.shape, x_test.shape

((3200, 11), (800, 11))

In [11]:
y_train.mean(), y_test.mean()

(np.float64(0.2653125), np.float64(0.265))

Logistic model

In [12]:
x_train_const = sm.add_constant(x_train)

model = sm.Logit(y_train, x_train_const).fit()

Optimization terminated successfully.
         Current function value: 0.199073
         Iterations 9


In [13]:
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:                  churn   No. Observations:                 3200
Model:                          Logit   Df Residuals:                     3188
Method:                           MLE   Df Model:                           11
Date:                Thu, 10 Sep 2026   Pseudo R-squ.:                  0.6559
Time:                        16:55:36   Log-Likelihood:                -637.03
converged:                       True   LL-Null:                       -1851.3
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                            3.9494      0.308     12.810      0.000       3.345       4.554
near_location                   -0.1004      0.186     -0.538      0.590     

Odds ratios

In [14]:
OD = np.exp(model.params)
print(OD)

const                            51.903980
near_location                     0.904490
partner                           0.961884
promo_friends                     0.649650
group_visits                      0.447501
avg_additional_charges_total      0.994722
lifetime                          0.353953
frequency_drop                  104.246875
contract_period_6                 0.214073
contract_period_12                0.052852
age_group_26-35                   0.151513
age_group_36+                     0.007087
dtype: float64


In [15]:
freq_OD = np.exp(model.params['frequency_drop'] * 0.1)
print(freq_OD)

1.5914987592361014


Testing

In [17]:
x_test_const = sm.add_constant(x_test)

In [18]:
predictions = model.predict(x_test_const)

In [23]:
predictions.head()

1799    0.002076
2296    0.409235
1177    0.883149
3375    0.074411
2206    0.000764
dtype: float64

In [24]:
y_pred = (predictions >= 0.5).astype(int)

In [26]:
y_pred.head()

1799    0
2296    0
1177    1
3375    0
2206    0
dtype: int64

In [28]:
conf_mtx = confusion_matrix(y_test, y_pred)
print(conf_mtx)

[[568  20]
 [ 37 175]]


In [31]:
cr = classification_report(y_test, y_pred)
print(cr)

              precision    recall  f1-score   support

           0       0.94      0.97      0.95       588
           1       0.90      0.83      0.86       212

    accuracy                           0.93       800
   macro avg       0.92      0.90      0.91       800
weighted avg       0.93      0.93      0.93       800



In [33]:
roc_auc = roc_auc_score(y_test, predictions)
print(roc_auc)

0.9719147092799384
